# Overture — per-row licensing (live)

Overture aggregates many sources with different licenses. The backend surfaces a per-row `license_id` and warns when share-alike `ODbL-1.0` (OpenStreetMap-derived) rows are present — critical for commercial redistribution.

In [1]:
LAT_LIM = [40.757, 40.759]  # [south, north]
LON_LIM = [-73.987, -73.984]  # [west, east] — a Times Square block
OUT = "_overture_out"

In [2]:
import warnings
from earthlens.earthlens import EarthLens
from earthlens.overture import LicenseWarning

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter('always')
    paths = EarthLens(
        data_source="overture",
        dataset="buildings", variables=[],
        aoi=[LON_LIM[0], LAT_LIM[0], LON_LIM[1], LAT_LIM[1]], path=OUT,
    ).download()

for w in caught:
    if issubclass(w.category, LicenseWarning):
        print('LicenseWarning:', w.message)

2026-05-27 14:49:52 | INFO | pyramids.base.config | Logging is configured.


2026-05-27 14:49:52.104 | INFO     | earthlens.overture.backend:_fetch:377 - Fetching Overture 'building' (theme 'buildings') for bbox (-73.987, 40.757, -73.984, 40.759) (release=latest)


2026-05-27 14:51:49.576 | INFO     | earthlens.overture.backend:_fetch:386 - buildings/building: wrote 45 feature(s) to C:\gdrive\algorithms\remote-sensing\earthlens\.claude\worktrees\overture\docs\examples\overture\_overture_out\overture_buildings_building_latest.parquet


In [3]:
import geopandas as gpd

gdf = gpd.read_parquet(paths[0])
gdf['license_id'].value_counts()

license_id
ODbL-1.0    45
Name: count, dtype: int64

The derivation rule is exposed directly as `row_license` for ad-hoc analysis of a `sources` cell — ODbL wins whenever an OSM source is present, even if it is not listed first.

In [4]:
from earthlens.overture import row_license

print(row_license([
    {'dataset': 'Overture', 'license': 'CDLA-Permissive-2.0'},
    {'dataset': 'OpenStreetMap', 'license': 'ODbL-1.0'},
]))
print(row_license([{'dataset': 'Foursquare', 'license': 'Apache-2.0'}]))

ODbL-1.0
Apache-2.0
